### Notebook 范围

### 目的
对同年同初始化月的 INT 与 NOCOUPL small-perturbation case 做配对差值。

### 科学问题
interactive O3 是否系统性改变 O3 loss、vortex wind、EP100 W1+W2 或 vortex persistence？

### 方法
变量为 INT-NOCOUPL：O3 partial column、U60N10、U60N50、EP100 W1+W2。成员 ID 能配对时做 member-paired difference，否则比较 ensemble mean。

### 输出
outputs/figures/07_int_vs_nocoupl/<year>/ 和 outputs/tables/07_int_vs_nocoupl。

### 解读
若 INT-NOCOUPL 在多个 case 中同号，说明 ozone-radiative feedback 对 vortex/O3 pathway 有系统影响。

### 注意
0003 没有可靠 paired NOCOUPL 时不纳入配对反馈检验。


### 导入与公共路径

### 目的
加载 Extention_analysis 公共函数。

### 科学问题
所有 notebook 是否共享相同路径、变量定义和 sign convention？

### 方法
导入 hindcast_extension_utils.py。

### 输出
无图。

### 解读
路径正确即可继续。

### 注意
所有输出都写入 outputs 子目录。


In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import hindcast_extension_utils as hx
from hindcast_extension_utils import *

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)


### 绘制 INT minus NOCOUPL evolution

### 目的
为每个 paired case 绘制 O3、U60N10、U60N50 的时间差值和 EP100 W1+W2 成员差。

### 科学问题
interactive O3 是否维持更低 O3、更强/更持久 vortex 或改变 wave forcing？

### 方法
时间序列 panel 是 ensemble mean difference，阴影为成员差 ±1 std；EP100 panel 是 source window 的成员差。x 轴为月份。

### 输出
07_INT_minus_NOCOUPL_evolution_<year>_<init>_v2.png/pdf 和总览表。

### 解读
正/负号需要结合变量解释：O3 负值表示 INT 更低臭氧；U 正值表示 INT 更强西风；FWD 在 08 单独检验。

### 注意
不同 case 起始月份不同，不解释初始化前过程。


In [ ]:
fig_base = figure_dir("07_int_vs_nocoupl")
tab_dir = table_dir("07_int_vs_nocoupl")
inv = discover_hindcast_cases(); pairs = paired_int_nocoupl_cases(inv)
rows = []

def _paired_diff_da(a, b):
    ma = [member_short_id(v) for v in a["member"].values]
    mb = [member_short_id(v) for v in b["member"].values]
    common = [m for m in ma if m in set(mb)]
    if not common:
        return None
    ia = [ma.index(m) for m in common]; ib = [mb.index(m) for m in common]
    aa = a.isel(member=ia)
    bb = b.isel(member=ib)
    time_dim = "lead_time" if "lead_time" in aa.dims else "time"
    ntime = min(aa.sizes[time_dim], bb.sizes[time_dim])
    aa = aa.isel({time_dim: slice(0, ntime)})
    bb = bb.isel({time_dim: slice(0, ntime)})
    data = np.asarray(aa.values, dtype=float) - np.asarray(bb.values, dtype=float)
    coords = {"member": common, time_dim: aa[time_dim].values}
    out = xr.DataArray(data, dims=("member", time_dim), coords=coords, name="INT_minus_NOCOUPL")
    if "date" in aa.coords:
        out = out.assign_coords(date=(time_dim, aa["date"].values[:ntime]))
    return out

for _, pair in pairs.iterrows():
    icase = pair["INT_case"]; ncase = pair["NOCOUPL_case"]; year = pair["year"]; init = pair["init_month"]
    fig_dir = fig_base / str(year)
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(4, 1, figsize=(10.5, 9.2), sharex=False, constrained_layout=True)
    case_rows = []
    start_doy = init_doy_for_case(icase); end_doy = mmdd_to_doy(5, 30)
    for ax, varname, loader, ylabel, color in [
        (axes[0], "O3", lambda c: load_hindcast_o3(c), "ΔO3 INT-NOCOUPL (DU)", "tab:blue"),
        (axes[1], "U60N10", lambda c: compute_u60(c, 10), "ΔU60N10 (m/s)", "tab:orange"),
        (axes[2], "U60N50", lambda c: compute_u60(c, 50), "ΔU60N50 (m/s)", "tab:green"),
    ]:
        ai, di = loader(icase); an, dn = loader(ncase)
        if ai is None or an is None:
            ax.text(0.5, 0.5, f"{varname} missing", transform=ax.transAxes, ha="center")
            continue
        diff = _paired_diff_da(ai, an)
        if diff is None:
            ax.text(0.5, 0.5, f"{varname}: no paired members", transform=ax.transAxes, ha="center")
            continue
        x = case_time_doy(icase, di[: diff.sizes["lead_time"]])
        mean = diff.mean("member", skipna=True); std = diff.std("member", skipna=True)
        ax.plot(x, mean.values, color=color, lw=2.2, label="paired mean")
        ax.fill_between(x, (mean-std).values, (mean+std).values, color=color, alpha=0.2, label="±1σ member diff")
        ax.axhline(0, color="0.2", lw=0.8)
        ax.set_ylabel(ylabel)
        set_month_axis(ax, start_doy, end_doy, label="")
        case_rows.append({"year": year, "init_month": init, "INT_case": icase, "NOCOUPL_case": ncase, "variable": varname, "mean_delta": float(mean.mean(skipna=True))})
    ep_i = compute_all_ep100(icase, source_windows_for_case(icase)["primary"])
    ep_n = compute_all_ep100(ncase, source_windows_for_case(ncase)["primary"])
    if not ep_i.empty and not ep_n.empty:
        ei = ep_i.set_index("member"); en = ep_n.set_index("member")
        common = [m for m in ei.index if m in set(en.index)]
        if common:
            delta = ei.loc[common, "EP100_wave1_plus_wave2"] - en.loc[common, "EP100_wave1_plus_wave2"]
            axes[3].bar(np.arange(len(delta)), delta.values, color="tab:purple", edgecolor="black")
            axes[3].axhline(0, color="0.2", lw=0.8)
            axes[3].set_ylabel("ΔEP100 W1+W2")
            axes[3].set_xlabel("paired member")
            axes[3].set_title("EP100 W1+W2 source-window member difference\nmean(-ep2), 100 hPa, 40-80N; positive = stronger INT upward wave activity")
            case_rows.append({"year": year, "init_month": init, "INT_case": icase, "NOCOUPL_case": ncase, "variable": "EP100_W1W2", "mean_delta": float(delta.mean())})
    fig.suptitle(f"INT - NOCOUPL paired evolution: {year}-{init}")
    case_csv = tab_dir / f"07_INT_minus_NOCOUPL_evolution_{year}_{init}.csv"
    pd.DataFrame(case_rows).to_csv(case_csv, index=False)
    rows.extend(case_rows)
    savefig(fig, f"07_INT_minus_NOCOUPL_evolution_{year}_{init}_v2", fig_dir=fig_dir, notebook="07_INT_vs_NOCOUPL_feedback.ipynb", scientific_question="interactive O3 是否改变 O3/vortex/EP pathway？", variables_windows="INT-NOCOUPL paired members; O3/U init-May30; EP100 W1+W2 primary source window", interpretation="多 case 同号差值支持系统性 ozone feedback。", caveat="仅 small perturbation paired cases；0003 不强行配对。", csv_table=case_csv)
    plt.close(fig)
metrics = pd.DataFrame(rows)
metrics_csv = tab_dir / "07_INT_minus_NOCOUPL_evolution_metrics.csv"
metrics.to_csv(metrics_csv, index=False)
metrics.to_csv(TAB_DIR / "07_INT_minus_NOCOUPL_evolution_metrics.csv", index=False)
print(metrics.to_string(index=False))
write_figure_guide()
